# Drug Interaction Prediction

You're an ML engineer at a pharmaceutical research lab. The lab wants to predict drug interactions and their severity — given a pair of drugs and some patient information, can we predict whether the drugs will interact, and how severely?

The data is complex: molecular features, drug categories, patient demographics. The classes are imbalanced — most drug pairs don't interact at all, and the rare severe interactions are the ones we most need to catch. Getting this wrong has real consequences.

This is the most technically demanding lab so far, and it demands **experimental rigour**, not just good results.

## Loading and exploring the data

In [ ]:
%pip install seaborn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, f1_score, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
df = pd.read_csv("./drug_interactions.csv")
print(f"Shape: {df.shape}")
df.head()

In [ ]:
print("Target distribution:")
print(df["interaction_severity"].value_counts())
print(f"\n{df['interaction_severity'].value_counts(normalize=True).round(3)}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
df["interaction_severity"].value_counts().reindex(["none", "mild", "moderate", "severe"]).plot(
    kind="bar", ax=ax, color=["steelblue", "gold", "orange", "coral"]
)
ax.set_title("Interaction severity distribution")
ax.set_ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

~55% of drug pairs have no interaction. Only ~5% are severe. This is heavily imbalanced multi-class — and the rare class (severe) is the one we most need to catch.

In [ ]:
print("Drug A categories:", df["drug_a_category"].unique())
print("Drug B categories:", df["drug_b_category"].unique())
print("Patient age groups:", df["patient_age_group"].unique())
print(f"\nNumerical summary:\n{df.describe().round(2)}")

## Preprocessing pipeline

This dataset has both numerical and categorical features. They need different preprocessing: numerical features get scaled, categorical features get one-hot encoded. scikit-learn's `ColumnTransformer` lets us define different transformations for different columns and apply them in a single step.

In [ ]:
numerical_features = ["drug_a_molecular_weight", "drug_a_solubility", "drug_a_half_life"]
categorical_features = ["drug_a_category", "drug_b_category", "patient_age_group"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

We wrap this in a `Pipeline` so preprocessing and modelling happen together. This is critical for cross-validation — without a pipeline, you'd risk data leakage (fitting the scaler on the full dataset before splitting).

In [ ]:
X = df[numerical_features + categorical_features]
y = df["interaction_severity"]

## Multi-class classification

So far you've done binary classification (two classes). Now there are four severity levels. The model must learn to distinguish between all four.

Multi-class cross-entropy extends binary cross-entropy: the model outputs a probability for each class, and the loss penalises the model for putting low probability on the correct class.

Most scikit-learn classifiers handle multi-class automatically — logistic regression uses a one-vs-rest or multinomial approach.

## Experiment design

Before training a single model, plan the experiment. Answer these questions:

1. **Which models will you compare?** We'll use three: logistic regression, random forest, and gradient boosting. These represent different approaches (linear vs ensemble vs boosting).
2. **What metric?** With imbalanced multi-class data, accuracy is misleading. We'll use **weighted F1** — it computes F1 for each class and averages weighted by class frequency.
3. **How will we ensure fair comparison?** Same preprocessing, same cross-validation splits, same random seeds. The only thing that changes is the model.

## Cross-validation

A single train/test split gives you one number. Is that number reliable? Maybe you got lucky (or unlucky) with the split. Cross-validation trains and evaluates the model k times, each time with a different split. Stratified k-fold ensures each fold maintains the class distribution.

In [ ]:
# Create a StratifiedKFold with 5 splits, shuffle=True, random_state=42
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## Model comparison

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, multi_class="multinomial", random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(n_estimators=100, random_state=42))
    ]),
    "Gradient Boosting": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", GradientBoostingClassifier(n_estimators=100, random_state=42))
    ]),
}

# Cross-validate each model and store the results
results = {}
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring="f1_weighted")
    results[name] = scores
    print(f"{name}: F1 = {scores.mean():.3f} ± {scores.std():.3f}")

In [ ]:
# Visualise the comparison
results_df = pd.DataFrame(results)
fig, ax = plt.subplots(figsize=(8, 5))
results_df.boxplot(ax=ax)
ax.set_ylabel("Weighted F1 Score")
ax.set_title("Model comparison (5-fold cross-validation)")
plt.tight_layout()
plt.show()

## Statistical significance

Model A scored 0.82 ± 0.03. Model B scored 0.81 ± 0.04. Is A better than B? Maybe — but the confidence intervals overlap substantially. The difference might just be noise from different fold compositions.

Look at the box plot. If the boxes overlap heavily, the models are performing comparably. If one is clearly above the others with no overlap, you have a more confident winner.

This is the intuition behind statistical significance: not just "which number is bigger" but "is the difference bigger than the noise?" In a research context, you'd use a proper statistical test. Here, the visual comparison and standard deviation give you enough to make a practical decision.

## Hyperparameter tuning

Take the best-performing model and tune its hyperparameters. `GridSearchCV` tries every combination and finds the best.

In [ ]:
# Choose the best model from the comparison and define a parameter grid
# for GradientBoostingClassifier
param_grid = {
    "classifier__n_estimators": [50, 100, 200],
    "classifier__max_depth": [3, 5, 7],
    "classifier__learning_rate": [0.05, 0.1, 0.2],
}

# Create the pipeline for the best model
best_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", GradientBoostingClassifier(random_state=42))
])

# Run GridSearchCV
grid_search = GridSearchCV(
    best_pipeline, param_grid, cv=cv, scoring="f1_weighted"
)
grid_search.fit(X, y)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best F1: {grid_search.best_score_:.3f}")

## Reproducibility

In [ ]:
# Full classification report on the best model
best_model = grid_search.best_estimator_

# Final evaluation with a held-out set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=["none", "mild", "moderate", "severe"])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["none", "mild", "moderate", "severe"])
fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(ax=ax, cmap="Blues")
plt.title("Final model confusion matrix")
plt.tight_layout()
plt.show()

### Thinking about reproducibility

- Could someone reproduce this experiment from your notebook alone?
- What would they need? The same data, the same random seeds, the same library versions.
- What's missing? No experiment tracking tool, no version pinning, no automated pipeline.
- In a real research lab, you'd use MLflow or Weights & Biases to log every experiment automatically. The notebook captures the process, but a tracking tool captures the history.